# mode_conditioned_rebound_reward_train_v2

## Purpose

Continue the overfit-control objective with a training-time, state-conditioned reward experiment. The previous `state_conditioned_actor_selector_v1` notebook showed that routing among existing all-WF actors does not clear the strict + walk-forward + 2025/2026 cluster gate, so this run changes the training objective rather than selection thresholds.

## External Basis

- DeepTrader: risk-return balanced portfolio RL with market-condition embedding and negative maximum-drawdown reward components. This supports training-time state conditioning instead of post-hoc selector routing.
  https://ojs.aaai.org/index.php/AAAI/article/view/16144
- Potential-based reward shaping preserves policy-optimality when used as a shaped potential rather than arbitrary hindsight tuning. This supports keeping PBRS mild and auditable.
  https://ai.stanford.edu/~ang/papers/shaping-icml99.ps
- Probability of Backtest Overfitting (PBO) motivates walk-forward and out-of-sample checks before accepting a high-return candidate.
  https://doi.org/10.21314/JCF.2016.322
- Qlib is retained as the framework boundary for repeatable training/backtesting.
  https://github.com/microsoft/qlib

## Experiment Governance

- Direction: `reward`
- Changed layer: `reward` + mode-conditioned training bands; no broker/runtime binding.
- Canonical baseline: `balanced_rule_residual_v22` / current governed runtime family.
- Direct comparison baseline: `drawdown_rebound_reward_train_v1`.
- Role: `ablate`, because the previous reward attempt was too defensive and this tests whether participation-balanced mode reward fixes that failure.
- Target windows: fixed split plus `wf_2020`, `wf_2022`, `wf_2023_2024`.
- Acceptance: strict target is excess >= 30pp and MaxDD <= 15% where feasible; governance gate for this reward branch is at least no worse than v1 on drawdown while materially improving 2020/2023-2024 underparticipation.
- Expected trade-off: slightly higher stress exposure than v1, but lower global underparticipation.
- Failure mode: same as v1 if fixed/test_proxy relative return remains deeply negative; overfit if valid improves but WF fails.

## Historical Reflection

- Closest success: `balanced_rule_residual_v22` has corrected walk-forward 2023-2024 evidence and remains the important single-profile frontier, but its absolute drawdown is still above the user's target.
- Closest failure: `drawdown_rebound_reward_train_v1` defended 2022 but collapsed in broad participation: fixed test_proxy relative return about -2026pp and only 1/3 WF windows passed.
- Why this is not a duplicate: v1 made global drawdown/CVaR fear dominant. v2 relaxes global drawdown fear, lowers flat-wait cap, raises trend/rebound participation rewards, and keeps the same walk-forward verification contract.


In [1]:
from pathlib import Path
import json
import subprocess
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "SPEC.md").exists():
            return candidate
    raise RuntimeError(f"repo root not found from {start}")

ROOT = find_repo_root(Path.cwd().resolve())
CARD = ROOT / "runtime/reward/card/mode_conditioned_rebound_reward_train_v2"
PROFILE = CARD / "profile.yaml"
LOG = CARD / "train_walkforward.log"
SUMMARY = ROOT / "runtime/model/card/mode_conditioned_rebound_reward_train_v2/soxl_mode_conditioned_rebound_reward_train_v2_summary.json"
WF_WORKERS = ROOT / "runtime/soxl/wf_workers"

print("card", CARD)
print("profile exists", PROFILE.exists())
print(PROFILE.read_text()[:1200])


card /Users/rziiiii/.openclaw/workspace/runtime/reward/card/mode_conditioned_rebound_reward_train_v2
profile exists True
name: mode_conditioned_rebound_reward_train_v2
max_iters: 8
episode_per_iter: 8
episode_len: 360
hidden_sizes: [128, 64]
lr: 0.0002
update_batch_size: 64
update_repeat: 6
reward:
  use_moderate_reward: true
  use_alpha_reward: true
  alpha_scale: 3.0
  use_kelly_alpha: true
  pbrs_coef: 0.18
  transaction_cost: 0.10
  turnover_penalty: 0.06
  smoothness_penalty: 0.04
  micro_rebalance_penalty: 0.50
  micro_rebalance_threshold: 0.05
  episode_drawdown_penalty: 3.0
  episode_drawdown_threshold: 0.12
  use_cvar_terminal: true
  cvar_terminal_coef: 0.75
  cvar_terminal_alpha: 0.10
  downside_asymmetry_coef: 1.35
  drawdown_penalty: 0.30
  downside_penalty: 0.20
  risk_budget_penalty: 0.24
  high_exposure_penalty: 0.06
  trend_alignment_bonus: 0.80
  trend_alignment_threshold: 0.02
  opportunity_cost_penalty: 0.16
  opportunity_threshold_pos: 0.42
  opportunity_threshold_

In [2]:
cmd = [
    sys.executable,
    "soxl_qlib_strategy_v2.py",
    "--profiles",
    "mode_conditioned_rebound_reward_train_v2",
    "--walk-forward",
    "--skip-data-refresh",
    "--seed",
    "42",
]
print("running", " ".join(cmd))
with LOG.open("w", encoding="utf-8") as handle:
    proc = subprocess.run(cmd, cwd=ROOT, stdout=handle, stderr=subprocess.STDOUT, text=True)
print("exit_code", proc.returncode)
print("log", LOG)
print("tail")
print("\n".join(LOG.read_text(encoding="utf-8", errors="replace").splitlines()[-80:]))
if proc.returncode != 0:
    raise SystemExit(proc.returncode)


running /Users/rziiiii/.openclaw/workspace/qlib_venv/bin/python soxl_qlib_strategy_v2.py --profiles mode_conditioned_rebound_reward_train_v2 --walk-forward --skip-data-refresh --seed 42


exit_code 0
log /Users/rziiiii/.openclaw/workspace/runtime/reward/card/mode_conditioned_rebound_reward_train_v2/train_walkforward.log
tail
[800:MainThread](2026-05-25 11:15:38,439) WARNING - qlib.BaseExecutor - [executor.py:121] - `common_infra` is not set for <qlib.backtest.executor.SimulatorExecutor object at 0x122e9fe50>
[800:MainThread](2026-05-25 11:15:38,443) INFO - qlib.backtest caller - [__init__.py:93] - Create new exchange
Exception ignored in: <function DataQueue.__del__ at 0x11c1d9940>
Traceback (most recent call last):
  File "/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/rl/utils/data_queue.py", line 148, in __del__
    self.cleanup()
  File "/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/rl/utils/data_queue.py", line 109, in cleanup
    if self._queue.empty():
       ^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.11/3.11.14_3/Frameworks/Python.framework/Versions/3.11/lib/python3.11/multiprocessin

In [3]:
def load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

summary = load_json(SUMMARY)
wf_payloads = {}
for name in ["wf_2020", "wf_2022", "wf_2023_2024"]:
    p = WF_WORKERS / f"mode_conditioned_rebound_reward_train_v2-{name}-seed42.json"
    wf_payloads[name] = load_json(p)

summary.keys(), list(wf_payloads)


(dict_keys(['profile', 'seed', 'feature_schema', 'position_cap', 'position_budget', 'selection_score', 'checkpoint_retention', 'model_config', 'sampling_config', 'warm_start', 'model_path', 'retained_checkpoints', 'valid', 'best_score', 'selection_history', 'slices', 'test_proxy', 'promotion']),
 ['wf_2020', 'wf_2022', 'wf_2023_2024'])

In [4]:
import pandas as pd

rows = []

def metric_row(label, payload):
    rows.append({
        "window": label,
        "return": payload.get("return"),
        "buy_and_hold_return": payload.get("buy_and_hold_return") or payload.get("bnh_return"),
        "relative_return": payload.get("relative_return") or payload.get("relative_outperformance"),
        "sharpe": payload.get("sharpe"),
        "max_dd": payload.get("max_dd"),
        "avg_pos": payload.get("avg_pos"),
        "stress_avg_pos": payload.get("stress_avg_pos"),
    })

metric_row("valid", summary.get("valid", {}))
metric_row("test_proxy", summary.get("test_proxy", {}))
for key in ["test_2020", "test_2022", "test_2023_2024"]:
    metric_row(key, summary.get("slices", {}).get(key, {}))
for key, payload in wf_payloads.items():
    metric_row(key, payload)

metrics = pd.DataFrame(rows)
metrics_path = CARD / "metrics_summary.csv"
metrics.to_csv(metrics_path, index=False)
metrics


,window,return,buy_and_hold_return,relative_return,sharpe,max_dd,avg_pos,stress_avg_pos
0,valid,20.567167,98.480336,-77.913169,1.047852,21.218932,0.307051,0.209682
1,test_proxy,496.780712,2542.027442,-2107.471934,0.991448,44.234213,0.267829,0.109228
2,test_2020,51.025766,355.736160,-304.710394,1.719022,21.434020,0.297235,0.081149
3,test_2022,-40.753958,-76.614257,65.015154,-1.507673,43.909386,0.131381,0.071337
4,test_2023_2024,45.591811,55.654474,-10.062662,0.993082,22.178461,0.261327,0.133451
5,wf_2020,50.083327,355.736160,-304.710394,1.402969,23.086304,NaN,NaN
6,wf_2022,-12.705282,-76.614257,61.694557,-0.751300,15.900697,NaN,NaN
7,wf_2023_2024,-16.085903,-39.952710,42.441868,-0.388753,30.489188,NaN,NaN


In [5]:
hist = pd.DataFrame(summary.get("selection_history", []))
hist_path = CARD / "selection_history.csv"
hist.to_csv(hist_path, index=False)

slice_rows = []
for key, payload in summary.get("slices", {}).items():
    row = {"window": key}
    row.update(payload)
    slice_rows.append(row)
slices = pd.DataFrame(slice_rows)
slices_path = CARD / "slice_metrics.csv"
slices.to_csv(slices_path, index=False)

selection = {
    "profile": "mode_conditioned_rebound_reward_train_v2",
    "summary_path": str(SUMMARY.relative_to(ROOT)),
    "profile_path": str(PROFILE.relative_to(ROOT)),
    "train_walkforward_log": str(LOG.relative_to(ROOT)),
    "decision_thresholds": {
        "target_relative_return_pp": 30.0,
        "target_max_dd_pct": 15.0,
        "governance_2022_max_dd_pct": 30.0,
        "governance_min_wf_relative_pp": 30.0,
    },
    "valid": summary.get("valid", {}),
    "test_proxy": summary.get("test_proxy", {}),
    "slices": summary.get("slices", {}),
    "walk_forward": wf_payloads,
    "promotion": summary.get("promotion", {}),
}

# Decision logic is intentionally strict for the user's target and separate from repo promotion.
def rel(row):
    return float(row.get("relative_return") if row.get("relative_return") is not None else row.get("relative_outperformance", 0.0))
def dd(row):
    return float(row.get("max_dd", 999.0))

strict_target_pass = all(
    rel(row) >= 30.0 and dd(row) <= 15.0
    for row in [summary.get("test_proxy", {}), summary.get("slices", {}).get("test_2022", {}), *wf_payloads.values()]
)
wf_relative_passes = sum(rel(row) >= 30.0 for row in wf_payloads.values())
wf_dd15_passes = sum(dd(row) <= 15.0 for row in wf_payloads.values())

selection["strict_user_target_pass"] = bool(strict_target_pass)
selection["wf_relative_passes"] = int(wf_relative_passes)
selection["wf_dd15_passes"] = int(wf_dd15_passes)
if strict_target_pass:
    decision = "candidate_for_deeper_validation"
elif rel(summary.get("test_proxy", {})) < 0.0:
    decision = "blocked_underparticipation_not_promoted"
elif wf_relative_passes < 2 or wf_dd15_passes < 2:
    decision = "diagnostic_no_promotion_wf_risk_gap"
else:
    decision = "diagnostic_no_promotion_target_gap"
selection["decision"] = decision

selection_path = CARD / "selection.json"
selection_path.write_text(json.dumps(selection, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps({k: selection[k] for k in ["decision", "strict_user_target_pass", "wf_relative_passes", "wf_dd15_passes"]}, indent=2))
print("wrote", metrics_path, hist_path, slices_path, selection_path)


{
  "decision": "blocked_underparticipation_not_promoted",
  "strict_user_target_pass": false,
  "wf_relative_passes": 2,
  "wf_dd15_passes": 0
}
wrote /Users/rziiiii/.openclaw/workspace/runtime/reward/card/mode_conditioned_rebound_reward_train_v2/metrics_summary.csv /Users/rziiiii/.openclaw/workspace/runtime/reward/card/mode_conditioned_rebound_reward_train_v2/selection_history.csv /Users/rziiiii/.openclaw/workspace/runtime/reward/card/mode_conditioned_rebound_reward_train_v2/slice_metrics.csv /Users/rziiiii/.openclaw/workspace/runtime/reward/card/mode_conditioned_rebound_reward_train_v2/selection.json


In [6]:
# Compare directly against v1 and v22 for the most relevant failure dimensions.
def load_summary(name):
    return load_json(ROOT / f"runtime/model/card/{name}/soxl_{name}_summary.json")

comparisons = []
for base_name in ["drawdown_rebound_reward_train_v1", "balanced_rule_residual_v22"]:
    base = load_summary(base_name)
    for window in ["test_proxy", "test_2020", "test_2022", "test_2023_2024"]:
        cur_payload = summary.get("test_proxy", {}) if window == "test_proxy" else summary.get("slices", {}).get(window, {})
        base_payload = base.get("test_proxy", {}) if window == "test_proxy" else base.get("slices", {}).get(window, {})
        comparisons.append({
            "baseline": base_name,
            "window": window,
            "delta_relative_return": rel(cur_payload) - rel(base_payload),
            "delta_max_dd": dd(cur_payload) - dd(base_payload),
            "delta_avg_pos": float(cur_payload.get("avg_pos", 0.0)) - float(base_payload.get("avg_pos", 0.0)),
        })
comparison = pd.DataFrame(comparisons)
comparison_path = CARD / "baseline_comparison.csv"
comparison.to_csv(comparison_path, index=False)
comparison


,baseline,window,delta_relative_return,delta_max_dd,delta_avg_pos
0,drawdown_rebound_reward_train_v1,test_proxy,-81.027650,6.557728,-0.013356
1,drawdown_rebound_reward_train_v1,test_2020,-7.646363,0.979297,-0.029827
2,drawdown_rebound_reward_train_v1,test_2022,9.024840,8.030840,-0.054085
3,drawdown_rebound_reward_train_v1,test_2023_2024,-6.708693,2.270403,-0.012199
4,balanced_rule_residual_v22,test_proxy,-2107.471934,-10.283768,-0.240694
5,balanced_rule_residual_v22,test_2020,-304.710394,5.211118,-0.117672
6,balanced_rule_residual_v22,test_2022,65.015154,11.198139,-0.138093
7,balanced_rule_residual_v22,test_2023_2024,-10.062662,-10.247228,-0.185482


## Runtime Conclusion

Decision: `blocked_underparticipation_not_promoted`.

This run is evidence against continuing the same reward-only mode-conditioned rebound family by small coefficient tweaks. Relaxing the global drawdown and CVaR fear while adding trend/rebound participation reward did not repair the core underparticipation problem.

Key results from `selection.json`:

| Window | Return | Buy-and-hold | Relative | MaxDD | Sharpe | Avg position |
|---|---:|---:|---:|---:|---:|---:|
| test_proxy | 496.8% | 2542.0% | -2107.5pp | 44.2% | 0.99 | 0.268 |
| test_2020 | 51.0% | 355.7% | -304.7pp | 21.4% | 1.72 | 0.297 |
| test_2022 | -40.8% | -76.6% | +65.0pp | 43.9% | -1.51 | 0.131 |
| test_2023_2024 | 45.6% | 55.7% | -10.1pp | 22.2% | 0.99 | 0.261 |
| wf_2020 | 50.1% | 355.7% | -304.7pp | 23.1% | 1.40 | 0.297 |
| wf_2022 | -12.7% | -76.6% | +61.7pp | 15.9% | -0.75 | 0.141 |
| wf_2023_2024 | -16.1% | -40.0% | +42.4pp | 30.5% | -0.39 | 0.255 |

Gate readout:

- Strict user target pass: `false`.
- Walk-forward relative-return pass count: `2/3` (`wf_2022`, `wf_2023_2024`).
- Walk-forward MaxDD <= 15% pass count: `0/3`.
- Fixed split promotion failed because `test_2022` MaxDD was `43.9%`, above the 30% governance guard.
- The broad `test_proxy` result remains dominated by missed upside: `-2107.5pp` relative return even with positive absolute return.

Interpretation:

- The experiment found a useful defensive/rebound trace in `wf_2022`, where relative return was `+61.7pp` and MaxDD was close to the 15% target at `15.9%`.
- That benefit did not transfer to strong or recovery windows. `wf_2020` still underparticipated by `-304.7pp`, and `wf_2023_2024` had negative absolute return with `30.5%` MaxDD.
- The failure is therefore not just insufficient training length. The reward family is still teaching a low-exposure policy in upside regimes while not reliably enforcing the desired drawdown cap.

Next valid loop:

Stop tuning this reward-only family unless a materially different training target is introduced. The next experiment should use poor-window / factor-difference evidence or behavior-cloning / actor-refresh supervision from the stronger v22/composer-style policy, with an explicit drawdown-cap objective. A coefficient-only v3 would be a duplicate of this failure mode.